Groundwater | Transport Modeling Track

# Step 8: Model Application — Spill → Capture at a Compliance Well

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → **8-Apply** → 9-Document → 10-Communicate

**Story so far:** In [04t_model_implementation.ipynb](04t_model_implementation.ipynb) you built the spill→capture SRC-pulse model on a refined corridor and verified the transport engine against an exact 2D analytical solution. In [05t_calibration.ipynb](05t_calibration.ipynb) you saw where calibration would fit (a thin bridge — the transport parameters stay locked). This notebook is the keystone application: you **use** the model to answer the question that has driven the whole track — **does a contaminant spill reach the compliance well above a stated threshold, and when?**

| Section | Time budget |
|:--|:--|
| **Keystone — spill → capture breakthrough + threshold verdict** | ~15–20 min |
| *Optional — Rung A (reactive: sorption-only and decay-only, tested separately) and Rung B (dispersivity sensitivity)* | +15–25 min |
| *Optional — Rung D (MF6 PRT capture zone: 2 MODFLOW flow solves + a bisection probe, ~30 s compute)* | +10–15 min |
| *Rung C (recirculation) — described only, not implemented* | ~5 min read |

**Navigation:** ← Back to [05t_calibration.ipynb](05t_calibration.ipynb)  ·  → Forward to your **written report** (Steps 9–10 are the report + presentation; there are no 06t/07t notebooks in the transport track).

---


## §0 — Orientation: what are we trying to find out?

**The scenario.** A contaminant spills as a known-mass, finite-duration pulse at a point roughly 90 m **upgradient** of a representative geothermal doublet. The doublet itself runs **flow-only** — a clean injection well and an extraction well shape a forced-gradient flow field, and no solute rides either well. The doublet's **extraction well is the compliance / monitoring well**: it does not carry the spill, but it does pump groundwater, and whatever the ambient groundwater carries (including an unrelated upgradient spill) arrives with it.

The question is blunt: **does the plume reach the compliance well above a stated absolute threshold, and when** — or does it dilute or decay below concern first? We fix a module-level constant `THRESHOLD_MGL` so the threshold is a single, clearly editable number.

### The model backbone

The model used throughout this notebook is built by a single helper, `transport_srcpulse_demo.build_srcpulse_demo()` — the *only* model backbone this notebook uses. It builds and runs a coupled steady-GWF / transient-GWT simulation on the corridor-refined grid you saw built in [04t](04t_model_implementation.ipynb): a finite **SRC** mass-loading pulse enters the aquifer at the spill point, migrates with the regional + doublet-forced flow field, and we read its breakthrough at the extraction well.

### What this notebook is — and is not

These 01t–08t notebooks are **ungraded demonstrations**: they teach how to set up a transport model, how one *could* calibrate one, and a few applications. They are **not** the graded deliverable — graded student work lives in `PROJECT/workspace/`, a separate track. Nothing here is scored; the checkpoints below are for your own understanding.

### What the model CAN and CANNOT tell you

| The model gives a **defensible** answer for… | The model is **not trustworthy** for… |
|:--|:--|
| **breakthrough / arrival** at the compliance well — whether and when the plume gets there | the **exact position of a concentration contour** |
| **threshold exceedance** — peak concentration vs. `THRESHOLD_MGL`, and the exceedance window | **how much mass, at what concentration**, is captured — PRT (below) tracks particles, not concentration |
| the **longitudinal (centerline) reach** of the plume down the corridor | anything that needs the *transverse* concentration field resolved from the ADE alone |
| the **mass balance** (SRC in vs. well/boundary out vs. storage/decay) | |
| **whether the spill footprint is captured, and how wide the doublet's capture zone is** — but only via MF6 PRT particle tracking (**Rung D**), never from the ADE concentration field | |

**Why the lateral claim fails — this is corrected physics.** On a grid that is **aligned** with the flow, MODFLOW 6's TVD advection scheme adds essentially **no transverse numerical dispersion** — a high transverse grid Péclet number by itself does not doom the lateral width (see [04t §5](04t_model_implementation.ipynb), which verifies this directly against the 2D analytical solution). The real reasons a lateral or contour claim is *not* defensible here are (a) the flow near a doublet is **convergent and oblique to the grid**, which *does* produce real cross-wind numerical dispersion, and (b) the **physical plume is sub-cell** near the source (α_T = 1 m is narrower than even the refined 10 m cells), so no feasible grid can *represent* it, regardless of the scheme. The honest tool for a lateral or capture-zone question is **MF6 PRT particle tracking**, not the smeared concentration field — and this notebook now builds exactly that, in **Rung D** below, rather than deferring it further. PRT resolves *which particles are captured*, not *how much mass at what concentration* — that distinction is the whole point of keeping it a separate rung.

### Section index
- **§1** — Setup: load the helper, set the threshold and toggles, build the default (conservative) spill
- **Keystone** — the spill → capture breakthrough and the threshold verdict · *checkpoint 1* (always runs)
- **Rung A** — reactive transport: two SEPARATE variants, sorption-only (R=2) and decay-only (toggle `RUN_REACTIVE`) · *checkpoint 2*
- **Rung B** — dispersivity sensitivity (toggle `RUN_ALPHA_L`) · *checkpoint 3*
- **Rung D** — PRT capture zone: is the spill footprint captured, and how wide is the capture zone? (toggle `RUN_PRT`) · *checkpoint 4*
- **Rung C** — recirculation (described only, not implemented)
- **§5** — Summary


In [ ]:
# §1 — Setup
import math
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../_SUPPORT/src')
sys.path.append('../../_SUPPORT/src/scripts/scripts_exercises')

from transport_srcpulse_demo import build_srcpulse_demo
from transport_prt_capture import (build_prt_capture, capture_halfwidth_at,
                                   WIDE_RELEASE_RADIUS_M, ABS_XY, INJ_XY)
from shared_functions import create_multiple_choice

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [ ]:
# Stated pedagogical compliance threshold (mg/L = g/m^3) -- edit this to explore other regulatory limits.
THRESHOLD_MGL = 1.0

# --- Toggles: optional/expensive rungs default to False (repo convention) ---
RUN_REACTIVE = False   # Rung A -- two SEPARATE variants: retardation (R=2) only, and first-order decay only
RUN_ALPHA_L  = False   # Rung B -- doubled longitudinal (and, per the locked ratio, transverse) dispersivity
RUN_PRT      = False   # Rung D -- MF6 PRT capture-zone particle tracking (spill footprint + wide capture-zone probe)

# --- Build + run the default (conservative) spill -> capture SRC-pulse demo (cached after the first run) ---
res = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=120.0, solubility_mgL=1000.0)

print(f"Doublet (flow-only)  : injection cell {res.inj_cell}, extraction/compliance cell {res.ext_cell}")
print(f"Spill (SRC pulse)    : {res.mass_g:.0f} g over {res.pulse_days:.0f} d  ->  {res.smassrate_gpd:.0f} g/d")
print(f"  spill location     : ({res.spill_xy[0]:.0f}, {res.spill_xy[1]:.0f})  (~90 m upgradient of the compliance well)")
print(f"Grid Peclet          : Pe_L {res.PeL_min:.1f}-{res.PeL_max:.1f} (<= 2 OK, longitudinal resolved)"
      f"   Pe_T {res.PeT_min:.0f}-{res.PeT_max:.0f} (not the whole lateral story -- see 04t sect. 5)")
print(f"Compliance threshold : THRESHOLD_MGL = {THRESHOLD_MGL:g} mg/L")


## Keystone — does the spill reach the compliance well above the threshold, and when?

This is the question the whole track has been building to. The default call above pins a **known mass** (3×10⁵ g), released as a **finite pulse** (30 days), migrating through the corridor-refined grid for 120 days. We read the concentration at the **extraction (compliance) well** over that time and compare it against `THRESHOLD_MGL`.

*(We do not re-verify the transport engine here — that verification against the exact 2D analytical solution already happened in [04t §5](04t_model_implementation.ipynb); this notebook only **applies** the already-verified model.)*

**Checkpoint 1** below asks you to predict the outcome *before* you see the plot — commit to an answer first.


In [ ]:
# Checkpoint 1 -- committed BEFORE we read the keystone breakthrough result.
create_multiple_choice('task_t08_checkpoint_1')


In [ ]:
# Keystone -- breakthrough at the compliance well, against the stated threshold.
t, c = res.times, res.breakthrough
peak, t_peak = res.peak_mgL, res.arrival_day
exceed = c > THRESHOLD_MGL
t_first = float(t[np.argmax(exceed)]) if exceed.any() else None
t_last = float(t[len(t) - 1 - np.argmax(exceed[::-1])]) if exceed.any() else None

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, c, color='firebrick', lw=2.2, label='concentration at the compliance well')
ax.axhline(THRESHOLD_MGL, color='0.4', ls='--', lw=1.4, label=f'threshold = {THRESHOLD_MGL:g} mg/L')
if t_first is not None:
    ax.axvspan(t_first, t_last, color='firebrick', alpha=0.08, label='exceedance window')
    ax.axvline(t_first, color='steelblue', ls=':', lw=1.4)
    ax.text(t_first, peak * 0.5, f'  first exceedance\n  ~day {t_first:.0f}',
            color='steelblue', fontsize=9, va='center')
ax.set_xlabel('time since spill [d]'); ax.set_ylabel('concentration [mg/L]')
ax.set_title('Breakthrough at the extraction (compliance) well')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

verdict = ("EXCEEDS the threshold" if exceed.any() else "stays BELOW the threshold")
print(f"Peak concentration    = {peak:.2f} mg/L at day {t_peak:.0f}")
if t_first is not None:
    print(f"Exceedance window     = day {t_first:.0f} to day {t_last:.0f} "
          f"(first crosses {THRESHOLD_MGL:g} mg/L at ~day {t_first:.0f}, last above it at ~day {t_last:.0f})")
print(f"Compliance verdict    : the plume reaches the well and {verdict}.")


In [ ]:
print("Mass balance (from the GWT budget) [g]:")
for k, v in res.mass_balance.items():
    # Guard against non-numeric entries (e.g. an "error" string on a
    # _mass_balance parse failure) -- the numeric keys are populated as
    # NaN in that case (see transport_srcpulse_demo._mass_balance), so
    # they still print cleanly with %.4g, and the real error text is
    # shown instead of crashing on 'Unknown format code g for str'.
    if isinstance(v, (int, float)):
        print(f"  {k:<30} {v:14.4g}")
    else:
        print(f"  {k:<30} {v}")
print()
print(f"Solubility guardrail: emergent source concentration {res.emergent_C_mgL:.1f} mg/L "
      f"vs solubility {res.solubility_mgL:.0f} mg/L  ->  "
      f"{'PASS' if res.solubility_ok else 'FAIL'} (x{res.solubility_margin:.1f} margin)")


**Keystone result.** The plume peaks at **≈4.95 mg/L around day 41** — comfortably clearing the stated `THRESHOLD_MGL = 1.0` mg/L threshold. This is a clear **EXCEEDANCE**: the compliance well sees the spill well above the regulatory limit, and it arrives within about six weeks of the release, not months or years. The mass balance closes to a small numerical imbalance and the emergent source-cell concentration stays safely below the stated solubility — so the numbers above are not an artefact of a broken budget or an over-concentrated source. ✅

This verdict — **reach, exceed, and roughly when** — is exactly the class of claim the grid supports (see §0): it is a longitudinal, flux-through-a-single-cell quantity, not a lateral or contour claim.


## Rung A — reactive transport: do sorption and decay, tested SEPARATELY, change the verdict? (`RUN_REACTIVE`)

The keystone above is **conservative**: no sorption, no decay. Real contaminants often sorb to the aquifer matrix and/or degrade. This rung reruns the same spill with two reactive variants, so you can see how much protection (if any) reactions buy you.

**Reason in R, not K_d.** The helper's `R` parameter is the **retardation factor** — the ratio of groundwater velocity to the (slower) plume velocity. Internally it converts your chosen `R` into a distribution coefficient, `Kd = (R - 1) * n / rho_b`, and passes that to MODFLOW 6's MST sorption package; you never have to guess a K_d value directly. We use `R = 2.0` and a first-order decay rate `lam = math.log(2) / 30` (a 30-day half-life).

**Checkpoint 2** below asks you to predict the shape of the effect (later? lower? both?) before the toggled rerun.


In [ ]:
# Checkpoint 2 -- committed BEFORE the (toggled) reactive rerun.
create_multiple_choice('task_t08_checkpoint_2')


In [ ]:
# Rung A (optional, toggled) -- retardation (R=2) and decay-only (lambda), overlaid vs. the conservative keystone.
if RUN_REACTIVE:
    R_VAL = 2.0
    LAM = math.log(2) / 30.0   # 30-day half-life
    # The longer window (220 d vs. the keystone's 120 d) is NOT because R=2
    # needs it to arrive -- it arrives at ~day 61, well inside 120 d -- it is
    # to capture the full retarded TAIL / exceedance behaviour after the peak.
    res_R = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=220.0,
                                 solubility_mgL=1000.0, R=R_VAL)
    res_decay = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=120.0,
                                     solubility_mgL=1000.0, lam=LAM)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, c, color='firebrick', lw=2.0, label=f'conservative (peak {peak:.2f} mg/L @ day {t_peak:.0f})')
    ax.plot(res_R.times, res_R.breakthrough, color='darkorange', lw=2.0,
            label=f'R={R_VAL:g} sorption only (peak {res_R.peak_mgL:.2f} mg/L @ day {res_R.arrival_day:.0f})')
    ax.plot(res_decay.times, res_decay.breakthrough, color='seagreen', lw=2.0,
            label=f'decay only, half-life 30 d (peak {res_decay.peak_mgL:.2f} mg/L @ day {res_decay.arrival_day:.0f})')
    ax.axhline(THRESHOLD_MGL, color='0.4', ls='--', lw=1.4, label=f'threshold = {THRESHOLD_MGL:g} mg/L')
    ax.set_xlabel('time since spill [d]'); ax.set_ylabel('concentration [mg/L]')
    ax.set_title('Rung A -- reactive transport vs. the conservative keystone')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"Conservative : peak {peak:.2f} mg/L @ day {t_peak:.0f}")
    print(f"R={R_VAL:g} only    : peak {res_R.peak_mgL:.2f} mg/L @ day {res_R.arrival_day:.0f}  "
          f"(Kd = {res_R.Kd:.3g} m^3/kg)")
    print(f"decay only    : peak {res_decay.peak_mgL:.2f} mg/L @ day {res_decay.arrival_day:.0f}  "
          f"(lambda = {res_decay.lam:.4g} 1/d)")
    for label, r in (("R=2", res_R), ("decay", res_decay)):
        still_exceeds = bool(np.any(r.breakthrough > THRESHOLD_MGL))
        print(f"  {label}: {'still EXCEEDS' if still_exceeds else 'now stays BELOW'} "
              f"the {THRESHOLD_MGL:g} mg/L threshold")
        if r.meta.get("peak_at_last_step"):
            print(f"  WARNING ({label}): the peak is still rising at the end of the "
                  "simulated window -- extend total_days before reading an arrival time.")
else:
    print("RUN_REACTIVE = False. Set it to True to rerun the spill with the sorption-only "
          "(R=2) and decay-only variants and confirm your checkpoint-2 prediction.")


**Rung A punchline.** Rung A is two SEPARATE reactive-transport variants, not one combined effect, and they behave differently. **Retardation (`R = 2.0`) alone** both **delays and lowers** the peak: it drops to **≈2.99 mg/L at ≈day 61** (later and lower than the conservative 4.95 mg/L @ day 41), because retardation slows the plume's own velocity by a factor of R while sorption also partitions mass onto the solid phase. **Decay alone** (30-day half-life) *lowers* the peak to **≈2.80 mg/L** but leaves the arrival **essentially unchanged** (day ≈40 vs. the conservative ≈41 — if anything marginally *earlier*, since later-arriving mass has had longer to decay in transit): decay removes mass, it does not slow the plume. Neither variant, at these parameter values, brings the peak below `THRESHOLD_MGL = 1.0` mg/L: you still have an exceedance either way. Retardation buys you *time* as well as some attenuation; decay buys you attenuation only, not time — the two are different questions, and a real risk assessment has to check both.


## Rung B — dispersivity sensitivity: what if α_L were larger? (`RUN_ALPHA_L`)

`α_L = 10 m` is the track's locked longitudinal dispersivity, but it is a calibrated best estimate, not a certainty. This rung reruns the spill with `alpha_L = 20.0` — double the default. **Important:** the helper preserves the course's locked 10:1 anisotropy ratio, so it scales **both** dispersivities together: doubling α_L also doubles the transverse dispersivity α_T (1 m → 2 m). That matters, because more transverse spreading means more dilution of the same released mass.

**Checkpoint 3** below asks you to predict how this reshapes the *pulse* (not a steady plateau — this is a finite-duration release with a toe, a peak, and a tail) before the toggled rerun.


In [ ]:
# Checkpoint 3 -- committed BEFORE the (toggled) alpha_L rerun.
create_multiple_choice('task_t08_checkpoint_3')


In [ ]:
# Rung B (optional, toggled) -- alpha_L doubled (alpha_T doubles too, per the locked 10:1 ratio).
if RUN_ALPHA_L:
    res_aL = build_srcpulse_demo(mass_g=3.0e5, pulse_days=30.0, total_days=120.0,
                                  solubility_mgL=1000.0, alpha_L=20.0)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(t, c, color='firebrick', lw=2.0,
            label=f'alpha_L=10 m (default; peak {peak:.2f} mg/L @ day {t_peak:.0f})')
    ax.plot(res_aL.times, res_aL.breakthrough, color='steelblue', lw=2.0,
            label=f'alpha_L=20 m (peak {res_aL.peak_mgL:.2f} mg/L @ day {res_aL.arrival_day:.0f})')
    ax.axhline(THRESHOLD_MGL, color='0.4', ls='--', lw=1.4, label=f'threshold = {THRESHOLD_MGL:g} mg/L')
    ax.set_xlabel('time since spill [d]'); ax.set_ylabel('concentration [mg/L]')
    ax.set_title('Rung B -- effect of doubling alpha_L (and, per the locked ratio, alpha_T)')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f"alpha_L=10 m : peak {peak:.2f} mg/L @ day {t_peak:.0f}   "
          f"Pe_L {res.PeL_min:.1f}-{res.PeL_max:.1f}   Pe_T {res.PeT_min:.0f}-{res.PeT_max:.0f}")
    print(f"alpha_L=20 m : peak {res_aL.peak_mgL:.2f} mg/L @ day {res_aL.arrival_day:.0f}   "
          f"Pe_L {res_aL.PeL_min:.1f}-{res_aL.PeL_max:.1f}   Pe_T {res_aL.PeT_min:.0f}-{res_aL.PeT_max:.0f}")
    print(f"  alpha_T: {res.alpha_T:.1f} m -> {res_aL.alpha_T:.1f} m")
else:
    print("RUN_ALPHA_L = False. Set it to True to rerun the spill at 2x alpha_L "
          "and confirm your checkpoint-3 prediction.")


**Rung B punchline.** Doubling α_L to 20 m brings the toe **earlier**, broadens the front, and lengthens the tail. The peak also drops, to ≈4.21 mg/L at ≈day 38.8, against the default's 4.95 mg/L at day 41 — but the *why* is not fully separable from this one run: spreading the same finite pulse out longitudinally in time already lowers the peak on its own, and because this helper doubles α_T right along with α_L (the locked 10:1 ratio), transverse dilution also contributes. Both mechanisms act together here, and since the two dispersivities always move in lockstep in this helper, this single run **cannot apportion** how much of the drop is longitudinal spreading vs. transverse dilution. Both grid Péclet numbers (Pe_L and Pe_T) halve, since a larger dispersivity is easier for the same grid to resolve. The direction of the effect (lower, not higher) is a useful reminder that "more dispersion" does not automatically mean "more risk" at a fixed-mass release — it can dilute as much as it spreads.


## Rung D — PRT capture zone: is the spill footprint captured, and how wide is the doublet's capture zone? (`RUN_PRT`)

§0 and [04t §5/§6](04t_model_implementation.ipynb) have been deferring one question since the start of this track: the ADE model above resolves the **longitudinal** breakthrough at the compliance well to a percent or two, but near the doublet the flow crosses the refined grid at an oblique angle and the physical plume is sub-cell (α_T = 1 m, narrower than even the 10 m corridor cells) — so the ADE's **lateral** plume width and any concentration contour are *not* defensible (§0's CAN/CANNOT table). That leaves a real question unanswered: **is the spill's footprint actually captured by the extraction well, and how wide a swath of the aquifer feeds it?**

That is a **geometry / wellfield-protection** question, not a concentration question, and the ADE's smeared concentration field cannot answer it near a converging doublet. **MF6 PRT** (particle tracking) can: it is purely advective, so it is immune to the cross-wind numerical dispersion that corrupts the concentration field there. This rung pays that debt with `transport_prt_capture.build_prt_capture()` — a forward PRT run on the SAME doublet, SAME spill location, and SAME corridor-refined grid as the ADE model above (it does not build a new model from scratch). Two calls:

1. **The actual spill** — the realistic ~10 m footprint, `build_prt_capture()` — is it captured, and how long does it take?
2. **A wide capture-zone probe** — `release_radius_m=WIDE_RELEASE_RADIUS_M` (120 m), deliberately wide enough to straddle the capture-zone boundary, so it resolves *where the capture zone ends* rather than just confirming a small disc is captured.

**PRT answers "are advective particles captured" — nothing more.** It says nothing about concentration, dilution, reactions, or the finite duration of the spill; a 100%-captured footprint is not itself an exceedance claim. That verdict is, and remains, the ADE Keystone's job (above).

**Do not try to reconcile PRT's travel time with the ADE's peak by arithmetic.** PRT's median *advective* travel time (~26 d) and the ADE's *concentration* peak (day 41) are **different quantities**, and in a converging 2-D flow field no simple identity connects them. (The textbook identity *source centroid + mean advective travel time* predicts a breakthrough curve's **centre of mass**, not its **peak** — a dispersive BTC is right-skewed and peaks *before* its centroid — and even the centroid version fails here: it predicts 40.7 d against this breakthrough curve's actual centroid of 48.2 d.)

**What PRT *can* be verified against is the flow field it was handed.** Integrate the seepage velocity $v = q/n_e$ — specific discharge straight out of the GWF budget (`DATA-SPDIS`), the locked porosity $n_e = 0.20$ — along the spill→well axis, and compare it with what the particle tracker reports. That is an *independent computation of the same physics* (Eulerian flux vs Lagrangian particle integration), and the cell below runs it. **Be clear about its reach, though:** PRT and the ADE model share the **same flow field, the same grid and the same porosity** — only the *transport engines* differ. So this check verifies the **particle tracker** against the **flow solution**; it *cannot* detect an error in the flow solution itself. The track's real end-to-end transport verification is [04t §5](04t_model_implementation.ipynb)'s 2-D analytical benchmark.

**Checkpoint 4** below asks you to predict the footprint's capture verdict — and what it would (and would not) tell you — before the toggled run.


In [ ]:
# Checkpoint 4 -- committed BEFORE the (toggled) PRT capture-zone rerun.
create_multiple_choice('task_t08_checkpoint_4')


In [ ]:
# Rung D (optional, toggled) -- MF6 PRT: is the spill footprint captured, and where does the capture zone end?
if RUN_PRT:
    prt = build_prt_capture()                                             # spill footprint: r=10 m, 200 particles
    prt_wide = build_prt_capture(release_radius_m=WIDE_RELEASE_RADIUS_M)  # capture-zone PROBE disc: r=120 m

    # The REAL half-width: bisect the captured/escaped boundary on a TRANSECT (release single
    # particles at increasing |offset| and find the dividing streamline).  Unlike anything read
    # off the probe disc, this is a property of the FLOW FIELD -- it does not move when the probe
    # changes.  Measure it at the spill, and again 300 m upgradient, to watch the zone widen.
    hw_spill = capture_halfwidth_at(0.0)
    hw_up    = capture_halfwidth_at(-300.0, max_offset_m=200.0, n_scan=41)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # Panel 1 -- the actual spill: pathlines from the realistic 10 m footprint to the well.
    ax = axes[0]
    for i in range(prt.n_released):
        pl = prt.pathlines[prt.pathlines[:, 0].astype(int) == i]
        ax.plot(pl[:, 2], pl[:, 3], color='steelblue', lw=0.7, alpha=0.6)
    ax.scatter(*prt.spill_xy, color='firebrick', marker='*', s=200, zorder=5, label='spill footprint')
    ax.scatter(*ABS_XY, color='k', marker='v', s=90, zorder=5, label='extraction (compliance) well')
    ax.scatter(*INJ_XY, color='0.5', marker='^', s=90, zorder=5, label='injection well')
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_title(f'Panel 1 -- spill footprint pathlines\ncapture fraction {prt.capture_fraction:.3f}, '
                 f'median travel time {prt.tt_median_d:.1f} d')
    ax.legend(loc='best', fontsize=9)

    # Panel 2 -- the teaching object: WHICH release points are captured, and the bisected boundary.
    # The captured/escaped scatter is the real result here.  The scalar "fraction" printed below it
    # is a property of THIS DISC (see the takeaway), not of the doublet -- so it is not in the title.
    ax = axes[1]
    cap = prt_wide.captured
    ax.scatter(prt_wide.release_points[cap, 0], prt_wide.release_points[cap, 1],
               color='seagreen', s=16, label=f'captured ({prt_wide.n_captured})')
    ax.scatter(prt_wide.release_points[~cap, 0], prt_wide.release_points[~cap, 1],
               color='firebrick', s=16, label=f'escaped ({prt_wide.n_escaped})')
    u = np.array(prt_wide.axis_u); perp = np.array([-u[1], u[0]])
    sx, sy = prt_wide.spill_xy
    hp, hm = prt_wide.meta['halfwidth_plus_m'], prt_wide.meta['halfwidth_minus_m']
    ax.plot([sx - hm * perp[0], sx + hp * perp[0]], [sy - hm * perp[1], sy + hp * perp[1]],
            color='k', lw=2.4, zorder=6,
            label=f'bisected half-width at the spill ({prt_wide.halfwidth_at_spill_m:.0f} m)')
    ax.scatter(*prt_wide.spill_xy, color='k', marker='*', s=200, zorder=7, label='spill centre')
    ax.scatter(*ABS_XY, color='k', marker='v', s=90, zorder=5, label='extraction well')
    ax.scatter(*INJ_XY, color='0.5', marker='^', s=90, zorder=5, label='injection well')
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_title(f'Panel 2 -- fate of a {prt_wide.release_radius_m:.0f} m PROBE disc\n'
                 f'{prt_wide.n_captured}/{prt_wide.n_released} of THIS DISC captured '
                 f'(a property of the probe, not of the doublet)')
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout(); plt.show()

    print(f"Panel 1 -- spill footprint (r={prt.release_radius_m:.0f} m): {prt.n_released} of "
          f"{prt.n_particles} particles released "
          f"({prt.meta['n_dropped_wellcell']} dropped -- they landed inside the extraction-well cell,")
    print( "                                                                   where a particle would report a 0 d travel time)")
    print(f"  captured / escaped  = {prt.n_captured} / {prt.n_escaped}   capture fraction = {prt.capture_fraction:.3f}")
    print(f"  travel time [d]     = median {prt.tt_median_d:.1f}  (p10 {prt.tt_p10_d:.1f}, p90 {prt.tt_p90_d:.1f})")
    print()
    print(f"Panel 2 -- capture-zone PROBE disc (r={prt_wide.release_radius_m:.0f} m): {prt_wide.n_released} of "
          f"{prt_wide.n_particles} released ({prt_wide.meta['n_dropped_wellcell']} dropped in the well cell)")
    print(f"  captured / escaped  = {prt_wide.n_captured} / {prt_wide.n_escaped}   "
          f"fraction = {prt_wide.capture_fraction:.3f}  <-- a property of THIS DISC, not of the doublet:")
    print( "                        the disc reaches ~30 m DOWNGRADIENT of the well, and those points can never")
    print( "                        be captured -- yet they count as 'escaped'.  Read the scatter, not the scalar.")
    print(f"  max captured offset = {prt_wide.max_captured_offset_m:.1f} m  <-- the most off-axis point that HAPPENED to be")
    print( "                        captured in this disc: a LOWER BOUND that GROWS with the probe radius")
    print( "                        (82.9 m at r=120, 90.3 m at r=160, 86.9 m at r=200).  NOT a half-width.")
    print()
    print("THE LATERAL ANSWER -- a half-width that is a property of the FLOW FIELD, not of a probe:")
    print(f"  bisected half-width at the SPILL transect (s = 0)  = {hw_spill['halfwidth_m']:.1f} m "
          f"(+{hw_spill['halfwidth_plus_m']:.1f} / -{hw_spill['halfwidth_minus_m']:.1f} m -- the zone is NOT symmetric)")
    print(f"  bisected half-width 300 m UPGRADIENT of the spill  = {hw_up['halfwidth_m']:.1f} m  "
          f"(the capture zone WIDENS upgradient)")
    print(f"  analytic far-field asymptote  y_max = Q/(2 q b)    = {hw_spill['asymptotic_halfwidth_m']:.1f} m  "
          f"(q*b = {hw_spill['regional_qb_m2d']:.2f} m2/d, from the GWF budget)")
    print("      ^ that is 01t's screening formula y_max = Q/(2 T i), since T*i = K*b*i = q*b.  The numeric")
    print("        half-width AT THE SPILL is NARROWER than the far-field asymptote; far upgradient it reaches it.")
    print()
    print("VERIFICATION -- PRT's advection engine against the FLOW FIELD it was handed:")
    print(f"  PRT  path-averaged velocity (arc length / travel time) = {prt.v_prt_path_mpd:.2f} m/d   "
          f"(median arc {prt.arc_len_median_m:.1f} m in {prt.tt_median_d:.1f} d)")
    print(f"  FLOW q/n integrated along the spill->well axis         = {prt.v_flow_qn_mpd:.2f} m/d   "
          f"(travel-time integral {prt.tt_flow_integral_d:.1f} d over {prt.meta['spill_to_well_m']:.0f} m)")
    print("  -> the particle tracker reproduces the flow field's own seepage velocity to ~1%.  Two independent")
    print("     computations (Eulerian flux from the budget vs Lagrangian particle integration), nothing tuned.")
    print(f"     (The residual -- {prt.tt_flow_integral_d:.1f} d integral vs PRT's {prt.tt_median_d:.1f} d median -- is expected: "
          f"PRT's curved paths TERMINATE on")
    print(f"      entering the well CELL, so a particle flies {prt.arc_len_median_m:.1f} m, not the straight "
          f"{prt.meta['spill_to_well_m']:.0f} m axis.)")
    print()
    print(f"  NOTE: PRT's advective timescale (~{prt.tt_median_d:.0f} d) and the ADE's day-{t_peak:.0f} CONCENTRATION peak are")
    print("        DIFFERENT quantities.  No simple identity connects them in a converging 2-D field.")
else:
    print("RUN_PRT = False. Set it to True to run the PRT capture-zone rungs (spill footprint, capture-zone "
          "probe, and the transect half-width) and confirm your checkpoint-4 prediction.")


**Rung D takeaway.** The realistic ~10 m spill footprint is captured **completely** — capture fraction 1.000 (200/200), median advective travel time 25.8 d — so it sits entirely inside the doublet's capture zone; capture at this spill was never in doubt.

**The lateral answer: ≈79 m at the spill.** Widening the release to a 120 m *probe disc* shows the boundary exists, but the number to quote is not read off that disc. The defensible half-width is measured by **bisecting the dividing streamline on a transect**: release single particles at increasing lateral offset on the line through the spill, perpendicular to the spill→well axis, and find where capture flips to escape. That gives **≈79 m at the spill transect** (+81 m one side, −76 m the other — the zone is *not* symmetric, because the injection well sits off to one side). It is a property of the **flow field**: it does not move when the probe's radius, scan density or tolerance change. Far upgradient the zone **widens**, reaching ≈112 m at 300 m upgradient — converging on the flow field's own analytic asymptote **y_max = Q/(2·q·b) ≈ 108 m**, which is exactly [01t](01t_model_goal.ipynb)'s screening formula y_max = Q/(2·T·i). Two completely different computations of the same geometry, agreeing.

**Two numbers that are *not* the answer, and why.** (i) The **capture fraction 0.719** is a property of the **probe disc**, not of the doublet — it is "the fraction of a 120 m disc that happens to lie inside the capture zone". That disc even reaches ~30 m *downgradient* of the well, and those points can never be captured yet are counted as "escaped". Read the captured/escaped **scatter**; the scalar is an artefact of where we chose to release. (ii) The **maximum captured offset (82.9 m)** is a sampling statistic and a *lower bound*: in an unchanged flow field it grows with the probe (82.9 m at r=120, 90.3 m at r=160, 86.9 m at r=200), and the point that sets it sits 56 m *upgradient* of the spill, not on the spill transect. Neither is a capture-zone half-width. (Note also that 199, not 200, particles are released: one release point landed inside the extraction-well cell and was dropped, since such a particle would report a travel time of exactly 0 d. That dropped point *would* have been captured, so dropping it slightly **deflates** the fraction: 144/200 = 0.720 becomes 143/199 = 0.719.)

**The capture boundary is not a fence.** A PRT capture zone is a purely **advective** envelope. A real, dispersive plume straddles it: a source just *outside* the dividing streamline can still register at the well, at low concentration. "Outside 79 m ⇒ safe" is **false** — it is where the *advective* streamtube ends, not where risk ends.

**How the advection engine is verified.** Not against the ADE's day-41 peak — PRT's advective timescale (~26 d) and a concentration peak are different quantities, and no simple identity connects them in a converging 2-D field. Instead, PRT is checked against **the flow field it was handed**: integrating the seepage velocity v = q/n_e straight from the GWF budget (`DATA-SPDIS`, locked porosity 0.20) along the spill→well axis gives a path-averaged **3.21 m/d**, and PRT's particles report a path-averaged **3.24 m/d** (median arc length 83.6 m in 25.8 d) — agreement to ~1%, from two independent computations, with nothing tuned. (Note the arc length: the particles fly 83.6 m along a curved path and terminate on entering the well *cell*, so dividing the straight 90 m axis by the travel time would divide by a distance nothing travels.) Honest limit: PRT and the ADE share the **same flow field, grid and porosity** — only the transport engines differ — so this check verifies the *particle tracker*, and cannot detect an error in the *flow solution*. That end-to-end verification lives in [04t §5](04t_model_implementation.ipynb)'s 2-D analytical benchmark.

**The claim boundary, stated plainly.** Full advective capture of the spill footprint corroborates the Keystone's **arrival and timing** — the spill's water does reach this well, on an advective timescale of weeks — and nothing more. A capture fraction carries **no concentration information at all**, so it can never corroborate an *exceedance* verdict; that verdict is, and remains, the ADE model's alone. What full capture *does* buy the ADE is this: because the whole footprint stays inside the well's streamtube, the ADE's un-defensible **lateral** smearing cannot be diverting mass *out* of the well — so the lateral limitation does not undermine its peak-concentration claim. That supports the ADE without pretending PRT measures concentration.

## Rung C — recirculation (described, not implemented)

A different, related application question: for a geothermal doublet, **what fraction of the reinjected water (and any tracer riding it) short-circuits straight back to the extraction well?** This is the **recirculation** question from earlier versions of this track. It is a genuinely different scenario from spill→capture — reinjection carries tracer *through the doublet's own well*, not from an independent upgradient spill — so it is described here for completeness, not built.

<details>
<summary><b>What recirculation would ask, and what it would take to model it</b></summary>

**The question.** A GWHE doublet reinjects thermally spent water (carrying, in this framing, a conservative tracer at the injection concentration) at one well and extracts at another nearby well. Regional flow sweeps some of the reinjected water downgradient and away, but some fraction is drawn back to the extraction well by the pumping itself. That returned fraction degrades the thermal (or hydraulic) performance the installation is designed around, so quantifying it is operationally important.

**Why it is a different scenario, not a rung of this one.**
- The **source is the injection well itself**, continuously, not a one-off finite-mass pulse at an independent upgradient point.
- The relevant output is a **steady-state plateau ratio** (C∞ / c_inj at the extraction well) once the doublet reaches dynamic equilibrium, not a transient peak-and-decay breakthrough.
- The governing question is "how much of what I put IN comes back OUT of the same system," which is a mass-recycling question — spill→capture asks "does an EXTERNAL release reach a receptor," which is a threshold-and-arrival question. Conflating the two would blur both.

**What you would have to add to model it.** The current `transport_srcpulse_demo` backbone is flow-only at both doublet wells (no `SSM` auxiliary concentration on either `WEL` package) and uses a finite-pulse `SRC` term at an independent spill location. A live recirculation mode would need: (a) a continuous concentration on the **injection** well's `WEL` stress-period data (via the `AUXILIARY`/`AUX` concentration mechanism and a matching `SSM` entry, rather than `SRC` mass loading), (b) enough total simulated time to reach a steady plateau at the extraction well rather than a single transient peak, and (c) a plateau-reading diagnostic (e.g. the median of the last several breakthrough samples) in place of the peak/arrival-time diagnostics this notebook uses. None of that exists in the current helper, and building it is out of scope here.

</details>


## §5 — Summary

**The verdict.** For the default (conservative) spill — 3×10⁵ g released over 30 days, ~90 m upgradient of the compliance well — the plume **reaches the extraction well and exceeds** the stated threshold `THRESHOLD_MGL = 1.0` mg/L: peak **≈4.95 mg/L at ≈day 41**. Rung A's two separate variants both leave the exceedance in place at the parameter values explored here, but differently: retardation (sorption-only, R=2) delays AND damps it, while decay (decay-only) damps it without shifting the timing; a larger dispersivity (Rung B) shifts the peak earlier and slightly lower by spreading the finite pulse out longitudinally in time, with transverse dilution also contributing since this helper doubles α_T in lockstep with α_L (this single run cannot separate the two mechanisms). In every variant tested, the compliance well sees an exceedance — just at different times and magnitudes.

**Rung D closes the lateral question.** MF6 PRT particle tracking shows the realistic ~10 m spill footprint is **fully captured** (capture fraction 1.000, 200/200) with a median advective travel time of 25.8 d. The lateral number to stand behind is the **bisected capture-zone half-width — ≈79 m at the spill transect** (+81/−76 m; the zone is not symmetric), measured by finding the dividing streamline directly rather than reading a statistic off a probe disc, and therefore stable: it does not move with the probe. It **widens upgradient** to ≈112 m, converging on the flow field's analytic asymptote y_max = Q/(2·q·b) ≈ 108 m — the same screening formula [01t](01t_model_goal.ipynb) uses. The 120 m probe's capture fraction of 0.719 is *not* a physical result: it is a property of **that disc** (which even reaches downgradient of the well), and the 82.9 m maximum captured offset is a lower bound that grows with the probe radius. PRT's advection engine is verified against the **flow field it was handed** — the seepage velocity q/n_e integrated from the GWF budget along the spill→well axis gives 3.21 m/d against PRT's path-averaged 3.24 m/d — *not* against the ADE's day-41 peak, which is a different quantity that no simple identity connects to an advective travel time. Full capture corroborates the ADE's **arrival and timing**, and it also means the ADE's un-defensible lateral smearing is not diverting mass out of the well's streamtube. But a capture fraction carries **no concentration information**, so it can never corroborate the **exceedance** verdict — that is, and remains, the ADE model's alone. And a capture boundary is a purely *advective* envelope: a dispersive plume straddles it, so a source just outside 79 m is not thereby safe.

**What is defensible and what is not.** The breakthrough timing, peak concentration, threshold-exceedance window, longitudinal reach, and mass balance are all **longitudinal / flux-through-a-cell** quantities that the refined corridor resolves. Now, via Rung D's MF6 PRT particle tracking, **whether the spill footprint is captured and how wide the doublet's capture zone is** are defensible too — these are all claims you can stand behind. The **exact position of a concentration contour** is still not defensible from either model: not because the transport engine is wrong (it is verified against an exact analytical solution in [04t §5](04t_model_implementation.ipynb)), but because near a converging doublet the flow crosses the grid at an angle (real cross-wind numerical dispersion) and the physical plume is narrower than even the refined cells (a sub-cell representation limit) — and PRT, being advective-only, was never going to supply a concentration contour either. Reach, exceedance, and capture are now all on solid ground; only the concentration field's exact shape stays off the table.

**Reminder.** These 01t–08t notebooks are **ungraded demonstrations**. They exist to show you how a transport model is built, what it can tell you, and where its claims run out — not to produce a scored deliverable. Graded student work lives in `PROJECT/workspace/`.

**Navigation:** ← [05t_calibration.ipynb](05t_calibration.ipynb)  ·  → your **written report** (Steps 9–10).

> *Section-completion note:* this notebook intentionally does **not** call `create_section_completion_marker` (that tracker is bound to 02t). Use the per-section results above to track your progress.
